# ETL - Silver Layer

Esta camada representa a limpeza e transformação dos dados brutos da Bronze Layer. Aqui, aplicamos validações, correções e enriquecimentos para preparar os dados para análises mais avançadas.

## Objetivos:
- Limpar dados (remover nulos, duplicatas).
- Aplicar transformações (normalização, validações).
- Enriquecer dados com regras de negócio.
- Armazenar em formato Delta otimizado.

In [ ]:
# Importar bibliotecas necessárias
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

# Criar sessão Spark
spark = SparkSession.builder.appName("Silver Layer ETL").getOrCreate()

print("Sessão Spark criada com sucesso!")

In [ ]:
# Ler dados da Bronze Layer
bronze_df = spark.read.format("delta").load("/tmp/bronze/sample_data")

# Mostrar dados brutos
bronze_df.show(5)
print(f"Total de registros na Bronze: {bronze_df.count()}")

In [ ]:
# Limpeza e transformação: remover nulos, duplicatas e normalizar
silver_df = bronze_df \
    .dropDuplicates() \
    .na.drop(subset=["nome", "idade"]) \
    .withColumn("idade", col("idade").cast("int")) \
    .withColumn("cidade", lower(col("cidade"))) \
    .withColumn("categoria_idade", 
                when(col("idade") < 30, "jovem")
                .when(col("idade") < 40, "adulto")
                .otherwise("senior")) \
    .withColumn("processed_timestamp", current_timestamp())

# Mostrar dados transformados
silver_df.show(5)
print(f"Total de registros na Silver: {silver_df.count()}")

In [ ]:
# Salvar dados na camada Silver
silver_path = "/tmp/silver/sample_data_clean"
silver_df.write.format("delta").mode("overwrite").save(silver_path)

# Criar tabela Delta
spark.sql(f"""
CREATE TABLE IF NOT EXISTS silver_sample_data
USING DELTA
LOCATION '{silver_path}'
""")

print("Dados salvos na camada Silver com sucesso!")

In [ ]:
# Verificar tabela criada
spark.sql("DESCRIBE TABLE silver_sample_data").show()
spark.sql("SELECT * FROM silver_sample_data LIMIT 5").show()